In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
pd.read_csv("data/clean/test_scored.csv").to_sql("applicants", conn, index=False)
pd.read_csv("outputs/decline_reasons.csv").to_sql("declines", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

In [2]:
q("""
WITH d AS (
    SELECT score, default_2yrs,
           NTILE(10) OVER (ORDER BY score) AS decile
    FROM applicants)
SELECT decile,
       MIN(score) AS min_score, MAX(score) AS max_score,
       COUNT(*) AS applicants,
       ROUND(100.0 * AVG(default_2yrs), 1) AS bad_rate
FROM d
GROUP BY decile
ORDER BY decile
""")

,decile,min_score,max_score,applicants,bad_rate
0,1,403.0,538.0,4500,36.4
1,2,538.0,558.0,4500,11.7
2,3,558.0,571.0,4500,6.1
3,4,571.0,582.0,4500,4.3
4,5,582.0,591.0,4500,2.7
5,6,591.0,597.0,4500,2.3
6,7,597.0,603.0,4500,1.3
7,8,603.0,609.0,4500,1.0
8,9,609.0,613.0,4500,0.5
9,10,613.0,628.0,4500,0.6


In [3]:
q("""
SELECT CASE WHEN age < 30 THEN '18–29'
            WHEN age < 45 THEN '30–44'
            WHEN age < 60 THEN '45–59'
            ELSE '60+' END AS age_group,
       COUNT(*) AS applicants,
       ROUND(100.0 * AVG(CASE WHEN score >= 548 THEN 1 ELSE 0 END), 1) AS approval_rate,
       ROUND(100.0 * AVG(default_2yrs), 1) AS actual_bad_rate
FROM applicants
GROUP BY age_group
ORDER BY age_group
""")

,age_group,applicants,approval_rate,actual_bad_rate
0,18–29,2654,74.8,12.4
1,30–44,11755,79.5,9.7
2,45–59,16044,86.7,7.0
3,60+,14547,95.7,2.9


In [4]:
q("""
WITH cuts AS (
    SELECT 500 AS c UNION ALL SELECT 520 UNION ALL SELECT 548
    UNION ALL SELECT 570 UNION ALL SELECT 600)
SELECT c AS cutoff,
       ROUND(100.0 * SUM(score >= c) / COUNT(*), 1) AS approval_rate,
       ROUND(100.0 * SUM(CASE WHEN score >= c THEN default_2yrs ELSE 0 END) / SUM(score >= c), 2) AS bad_rate,
       ROUND((SUM(CASE WHEN score >= c THEN 1 - default_2yrs ELSE 0 END) * 400
            - SUM(CASE WHEN score >= c THEN default_2yrs ELSE 0 END) * 3000) * 10000.0 / COUNT(*)) AS profit_per_10k
FROM cuts CROSS JOIN applicants
GROUP BY c
ORDER BY c
""")

,cutoff,approval_rate,bad_rate,profit_per_10k
0,500,96.7,5.09,2194978.0
1,520,94.2,4.27,2399467.0
2,548,87.1,3.00,2595022.0
3,570,70.8,1.86,2384489.0
4,600,36.4,0.79,1358400.0


In [5]:
q("""
SELECT reason_1,
       COUNT(*) AS declines,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_declines,
       ROUND(AVG(points_short), 1) AS avg_points_short,
       ROUND(100.0 * AVG(default_2yrs), 1) AS bad_rate
FROM declines
GROUP BY reason_1
ORDER BY declines DESC
""")

,reason_1,declines,pct_of_declines,avg_points_short,bad_rate
0,High use of available credit limits,4270,73.3,29.6,28.1
1,Serious arrears (90+ days late),1087,18.7,56.4,46.8
2,Recent missed payments (30–59 days late),362,6.2,19.6,26.5
3,Missed payments (60–89 days late),108,1.9,14.3,25.9
